# TrainLens: diagnose a fine-tuning run and plan the next one

This quickstart uses a realistic train/validation/test result shape, plots the learning curves, runs TrainLens' deterministic analysis, and then optionally asks an OpenAI-compatible LLM to explain the run and propose the next experiments.

The example data is small and synthetic so the notebook runs immediately. In a real notebook, TrainLens can inspect the metrics and supported model/trainer objects already in memory.

In [ ]:
%pip install -q "trainlens[plots]"

## 1. Start from normal fine-tuning results

TrainLens keeps training, validation, and held-out test metrics separate. It also understands common aliases such as `eval_loss`, `val_loss`, `test_f1`, and split metric dictionaries.

In [ ]:
history = {
    "train_loss":     [1.31, 0.93, 0.66, 0.47, 0.34, 0.25],
    "val_loss":       [1.28, 0.91, 0.70, 0.58, 0.55, 0.57],
    "train_accuracy": [0.55, 0.68, 0.79, 0.87, 0.92, 0.95],
    "val_accuracy":   [0.54, 0.66, 0.75, 0.80, 0.82, 0.81],
}

test_metrics = {
    "loss": 0.64,
    "accuracy": 0.75,
    "f1": 0.74,
    "precision": 0.77,
    "recall": 0.72,
}

## 2. Plot the run and analyze it locally

The plot is optional (`trainlens[plots]`). The analysis itself does not call any external service.

In [ ]:
from trainlens import analyze, plot_training_curves, render_report

analysis = analyze(globals())
plot_training_curves(globals())
print(render_report(analysis))

For this run, TrainLens should preserve the final test F1/precision/recall, notice the train/validation generalization gap, and notice that the held-out test result is weaker than validation. With only validation metrics and no test metrics, it can instead recommend a final held-out evaluation.

On real fine-tuning objects, TrainLens can additionally report trainable/frozen parameters, PEFT adapters, LoRA rank/alpha/target modules, quantization, learning rates, and supported multimodal fine-tuning settings.

## 3. Optional: connect an LLM

Use `getpass()` so the key is not written into the notebook. Replace the base URL and model with any OpenAI-compatible provider you use. For a local Ollama/LM Studio/vLLM/llama.cpp endpoint, you can usually omit the API key.

In [ ]:
from getpass import getpass

from trainlens import OpenAICompatibleProvider

LLM_BASE_URL = "https://your-provider.example/v1"
LLM_MODEL = "your-model"

llm = OpenAICompatibleProvider.from_values(
    base_url=LLM_BASE_URL,
    model=LLM_MODEL,
    api_key=getpass("LLM API key: "),
)

### Define the analysis you want

For longer instructions, `prompt_text()` lets you write normal triple-quoted multiline text instead of quoting and concatenating every line. It removes the common indentation and outer blank space before the prompt is used.

The prompt below explicitly tells the LLM to explain the observed training behavior, keep test data as final evidence rather than a tuning target, and propose controlled next experiments.

In [ ]:
from trainlens import improvement_plan_prompt, prompt_text

prompt = improvement_plan_prompt(
    objective=prompt_text("""
        Explain what happened during this fine-tuning run and propose the three
        highest-value next experiments.

        Use the test split only as final generalization evidence; do not tune to it.
        Prefer controlled, low-cost changes first.
    """),
    model_family="LLM fine-tuning with PEFT/LoRA",
    focus_areas=(
        "train/validation generalization gap",
        "validation-to-test degradation",
        "checkpoint selection and early stopping",
        "learning-rate schedule",
        "regularization and adapter capacity",
    ),
)

### Keep reusable prompt instructions in `.md` files

TrainLens can also load prompt text directly from a UTF-8 Markdown file with `load_prompt()`. In a real project you can keep these files under a `prompts/` directory and version them independently from the notebook. This quickstart creates a small file first so the Colab remains self-contained.

In [ ]:
from pathlib import Path

from trainlens import load_prompt

prompt_file = Path("improvement_plan.md")
prompt_file.write_text(
    """Explain what happened during this fine-tuning run.

Prefer:
- controlled experiments
- low-cost changes first
- measurable success criteria
""",
    encoding="utf-8",
)

file_prompt = improvement_plan_prompt(
    objective=load_prompt(prompt_file),
    model_family="LLM fine-tuning with PEFT/LoRA",
)

print(file_prompt.objective)

### Preview the exact request before sending it

This performs no network request. `system_prompt` contains TrainLens' trusted instructions; `user_prompt` contains the sanitized notebook evidence.

In [ ]:
from trainlens import preview_llm_request

preview = preview_llm_request(
    globals(),
    mode="improvement_ideas",
    provider=llm,
    prompt_options=prompt,
)

print(preview.system_prompt)
print("\n--- SANITIZED EVIDENCE ---\n")
print(preview.user_prompt)

### Generate the explanation and next-step plan

The LLM receives the deterministic TrainLens findings and evidence. The improvement recipe asks it to return evidence-backed changes, expected effects, cost/risk, confidence, measurable success criteria, and one concrete next run.

In [ ]:
from trainlens import build_improvement_ideas

report = build_improvement_ideas(
    globals(),
    provider=llm,
    prompt_options=prompt,
)
report

## Local LLM alternative

If you run an OpenAI-compatible local server, replace the provider cell with:

```python
llm = OpenAICompatibleProvider.from_values(
    base_url="http://localhost:11434/v1",
    model="your-local-model",
)
```

Everything else in the notebook stays the same.